In [19]:
from tqdm import tqdm
from torchinfo import summary
import matplotlib.pyplot as plt
import data_proc.data_preproc as data_preproc
import dino.dino_features as features
import cutouts.cutouts as cutouts
import numpy as np
import torchvision.transforms as T

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import umap.umap_ as umap

In [20]:
import torch
from sklearn.neighbors import KNeighborsClassifier

In [21]:
import simclr.simclr as simclr

In [22]:
eddy_loader, eddy_loader_val = data_preproc.get_dataloaders(data = "eddy", size=256)

In [23]:
from sklearn.linear_model import LogisticRegression
def knn_score(X, y, X_val, y_val, k=5):
    neigh = KNeighborsClassifier(n_neighbors=k)
    neigh.fit(X, y)

    print(neigh.score(X_val, y_val))
    
def linear_probe_score(X, y, X_val, y_val, max_iter=2000, C=1.0):
    """
    Linear probe using multinomial logistic regression.

    X, X_val: [N, D] feature arrays
    y, y_val: [N] labels
    """
    
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-6

    Xn = (X - mu) / sigma
    Xvn = (X_val - mu) / sigma

    clf = LogisticRegression(
        max_iter=max_iter,
        C=C,
        solver="lbfgs",
        verbose=2
    )
    clf.fit(Xn, y)

    acc = clf.score(Xvn, y_val)
    print("linear probe accuracy:", acc)
    return acc, clf

In [24]:
import torch.nn as nn

def train_one_epoch(
    model: nn.Module,
    dataloader,
    transforms,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    device: str = "cuda",
    image_key: str = "image",
):
    """
    One training epoch.

    Args:
        model:        FrozenDinoSimCLR
        dataloader:   yields batches, typically dicts with batch[image_key]
        optimizer:    optimizer over trainable params only
        loss_fn:      your contrastive loss, called as loss_fn(zi, zj)
        device:       device string
        augment_fn:   function(batch_images) -> (xi, xj)
        image_key:    key for image tensor in batch dict
    """
    model.train()

    # Backbone should stay frozen and in eval mode
    model.backbone.eval()

    running_loss = 0.0
    num_batches = 0

    for batch in tqdm(dataloader, total=len(dataloader)):
        if isinstance(batch, dict):
            x = batch[image_key]
        else:
            x = batch[0]

        xi = transforms(x)
        xj = transforms(x)

        xi = xi.to(device, non_blocking=True)
        xj = xj.to(device, non_blocking=True)

        _, zi = model(xi)
        _, zj = model(xj)

        loss = loss_fn(zi, zj)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    return running_loss / max(num_batches, 1)

In [25]:
def frozen_features(loader, model):
    all_feats = []
    all_labels = []
    
    for batch in tqdm(loader, total=len(loader)):
        with torch.no_grad():
            proj = model.input_projector(batch["image"].to("cuda"))
            features = model.backbone(proj)
            features = features / features.norm(dim=-1, keepdim=True)

            all_feats.append(features.cpu())
            all_labels.append(batch["label"].cpu())

    X = torch.cat(all_feats, dim=0)
    y = torch.cat(all_labels, dim=0)

    return X, y

In [26]:
def check_knn_accuracy(model, loader, loader_val):
    X, y  = frozen_features(loader, model)
    X_val, y_val  = frozen_features(loader_val, model)
    
    knn_score(X, y, X_val, y_val, k=5)
    linear_probe_score(X, y, X_val, y_val, max_iter=2000, C=1.0)

In [27]:
def fit(
    model: nn.Module,
    train_loader,
    loader_val,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    epochs: int,
    device: str = "cuda",
    transforms=None,
    image_key: str = "image",
):
    model.to(device)

    for epoch in range(epochs):
        avg_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            transforms=transforms,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            image_key=image_key,
        )

        
        print(f"Epoch {epoch+1:03d} | loss={avg_loss:.6f}")

        if (epoch % 10 == 0):
            check_knn_accuracy(model, train_loader, loader_val)

In [28]:
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
backbone_feat_dim = 384



Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main


In [29]:
simclr_transforms_strong = T.Compose([
        T.RandomResizedCrop(64, scale=(0.3, 1.0)),
        T.RandomHorizontalFlip(),
        # transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3),
        # transforms.ToTensor()
    ])

simclr_transforms_weak = T.Compose([
        T.RandomResizedCrop(64, scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(),
        # transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        # T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
        # transforms.ToTensor()
    ])

In [12]:
model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=True,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

100%|██████████| 61/61 [00:31<00:00,  1.93it/s]


Epoch 001 | loss=3.913813


100%|██████████| 15/15 [00:06<00:00,  2.39it/s]


0.6242105263157894
linear probe accuracy: 0.6294736842105263


100%|██████████| 61/61 [00:31<00:00,  1.95it/s]


...

Epoch 041 | loss=3.320501


100%|██████████| 15/15 [00:06<00:00,  2.41it/s]


0.5821052631578948
linear probe accuracy: 0.6326315789473684


 51%|█████     | 31/61 [00:16<00:15,  1.93it/s]


KeyboardInterrupt: 

In [13]:
# not frozen dino
model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

100%|██████████| 61/61 [00:31<00:00,  1.97it/s]


Epoch 001 | loss=3.929295


100%|██████████| 15/15 [00:05<00:00,  2.52it/s]


0.68
linear probe accuracy: 0.6747368421052632


100%|██████████| 61/61 [00:31<00:00,  1.96it/s]


...

Epoch 052 | loss=3.305065


100%|██████████| 61/61 [00:31<00:00,  1.95it/s]


Epoch 053 | loss=3.327267


 62%|██████▏   | 38/61 [00:19<00:11,  1.95it/s]


KeyboardInterrupt: 

In [14]:
# No init on dino 
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 001 | loss=3.949191


100%|██████████| 15/15 [00:05<00:00,  2.51it/s]


0.6610526315789473
linear probe accuracy: 0.6463157894736842


100%|██████████| 61/61 [00:33<00:00,  1.82it/s]


...

0.6557894736842105
linear probe accuracy: 0.6347368421052632


100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 012 | loss=3.494940


 31%|███       | 19/61 [00:10<00:23,  1.79it/s]


KeyboardInterrupt: 

In [ ]:
# larger batch size better transforms 

# No init on dino 
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_weak

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:51<00:00,  1.17it/s]


Epoch 001 | loss=3.484747


100%|██████████| 15/15 [00:19<00:00,  1.33s/it]


0.5915789473684211
linear probe accuracy: 0.5821052631578948


100%|██████████| 61/61 [00:52<00:00,  1.17it/s]


...

100%|██████████| 61/61 [00:51<00:00,  1.18it/s]


Epoch 006 | loss=3.178318


100%|██████████| 61/61 [00:51<00:00,  1.18it/s]


Epoch 007 | loss=3.156460


 13%|█▎        | 8/61 [00:06<00:45,  1.17it/s]

# VIC REG

In [17]:
import torch.nn.functional as F

def vicreg_loss(z1, z2, sim_coeff=25, std_coeff=25, cov_coeff=1):
    # invariance
    sim_loss = ((z1 - z2)**2).mean()

    # variance
    std_z1 = torch.sqrt(z1.var(dim=0) + 1e-4)
    std_z2 = torch.sqrt(z2.var(dim=0) + 1e-4)

    std_loss = torch.mean(F.relu(1 - std_z1)) + \
               torch.mean(F.relu(1 - std_z2))

    # covariance
    z1 = z1 - z1.mean(dim=0)
    z2 = z2 - z2.mean(dim=0)

    cov_z1 = (z1.T @ z1) / (z1.size(0) - 1)
    cov_z2 = (z2.T @ z2) / (z2.size(0) - 1)

    off_diag = lambda x: x.flatten()[1:].view(x.size(0)-1, x.size(1)+1)[:, :-1]

    cov_loss = off_diag(cov_z1).pow(2).mean() + \
               off_diag(cov_z2).pow(2).mean()

    return sim_coeff * sim_loss + std_coeff * std_loss + cov_coeff * cov_loss

In [18]:
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=vicreg_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 001 | loss=45.848845


100%|██████████| 15/15 [00:06<00:00,  2.49it/s]


0.6578947368421053
linear probe accuracy: 0.6526315789473685


100%|██████████| 61/61 [00:33<00:00,  1.84it/s]


...

Epoch 009 | loss=45.674669


100%|██████████| 61/61 [00:33<00:00,  1.84it/s]


Epoch 010 | loss=45.663971


  7%|▋         | 4/61 [00:02<00:37,  1.52it/s]


KeyboardInterrupt: 